#**0. Imports**

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.datasets as dset
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Subset, DataLoader, ConcatDataset
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
from torch.utils.data import ConcatDataset

#**1. CIFAR-10 + ResNet Feature Extractor (UNCHANGED)**

In [ ]:
transform_resnet = T.Compose([
    T.ToTensor(),
    T.Resize((224,224), antialias=True),
    T.Normalize(mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225])
])

full_train_dataset = dset.CIFAR10(
    root="./data", train=True, download=True, transform=transform_resnet
)

all_labels = np.array(full_train_dataset.targets)

resnet = models.resnet18(weights="IMAGENET1K_V1")
resnet.fc = nn.Identity()
resnet.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = resnet.to(device)

#**2. Feature Extraction (UNCHANGED)**

In [ ]:
def extract_features_batched(dataset, batch_size=256):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    feats = []

    with torch.no_grad():
        for x,_ in loader:
            x = x.to(device)
            f = resnet(x)
            feats.append(f.cpu().numpy())

    return np.concatenate(feats, axis=0)

#**3. Imbalanced Data Distribution (WITHOUT replacement)**

In [ ]:
def generate_imbalanced_split(total, n_clients, alpha=0.5):
    """
    Generate an imbalanced split of `total` samples across n_clients.
    Lower alpha → more imbalance.
    """
    # Dirichlet gives proportions that sum to 1
    proportions = np.random.dirichlet(alpha=[alpha]*n_clients)
    #print(f"proportions: {proportions}")
    # Convert to counts
    raw_counts = (proportions * total).astype(int)
    #print(f"raw_counts: {raw_counts}")
    # Fix rounding issues
    diff = total - raw_counts.sum()
    raw_counts[np.argmax(raw_counts)] += diff
    #print(f"final_proportions: {proportions}")
    return raw_counts

In [ ]:
def rotate_list(lst, k):
    return lst[k:] + lst[:k]

In [ ]:
def distribute_cifar_imbalanced(labels, n_clients, alpha=0.5):
    """
    Distribute CIFAR-10 data across n_clients with class-wise imbalance.
    Returns dict: client_id → list of sample indices
    """

    client_indices = {i: [] for i in range(n_clients)}

    for cls in range(10):
        cls_idx = np.where(labels == cls)[0]
        np.random.shuffle(cls_idx)

        # Generate imbalance for this class
        base_split = generate_imbalanced_split(
            total=len(cls_idx),
            n_clients=n_clients,
            alpha=alpha
        )

        # Rotate per class (important!)
        split = rotate_list(list(base_split), cls % n_clients)
        print(f"cls: {cls} base_split: {base_split} split_after_rotate: {split}")

        start = 0
        for cid in range(n_clients):
            count = split[cid]
            client_indices[cid].extend(
                cls_idx[start:start+count]
            )
            start += count

        assert start == len(cls_idx), "Class split mismatch!"

    return client_indices

#**4. Initialize Clients**

#**5. Local & Global Distributions and Normalization**

In [ ]:
def compute_LD(y, n_classes=10):
    """
    Returns raw class-count vector for a client
    LD = [C0, C1, ..., C9]
    """
    return np.bincount(y, minlength=n_classes)

def compute_GD(LDs):
    """
    LDs: dict {cid -> raw LD vector}
    GD = sum of all LDs
    """
    return np.sum(list(LDs.values()), axis=0)

def normalize(dist):
    """
    Converts count vector to probability distribution
    """
    total = dist.sum()
    if total == 0:
        return dist
    return dist / total

#**6. Dominant Client Selection**

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def find_dominant_client_with_saturation(LDs, GD, clients, excluded_clients):
    sims = {}

    for cid, LD in LDs.items():
        if cid in excluded_clients:
            continue

        sims[cid] = cosine_similarity(
            LD.reshape(1, -1),
            GD.reshape(1, -1)
        )[0, 0]

    if len(sims) == 0:
        return None, {}

    dom = max(sims, key=sims.get)
    return dom, sims

#**7. Flicker Oversampling (UNCHANGED LOGIC)**

In [ ]:
def flicker_oversample(X_client, y_client, lin_thres=0.5):
    class_counts = Counter(y_client)
    Lmax = max(class_counts.values())

    new_X = [X_client]
    new_y = list(y_client)

    for cls, cnt in class_counts.items():
        Lin = cnt / Lmax
        if Lin < lin_thres:
            N_add = int(np.ceil(1 / Lin))
            cls_idx = np.where(y_client == cls)[0]
            chosen = np.random.choice(cls_idx, N_add, replace=True)

            base_ds = dset.CIFAR10(
                root="./data", train=True, download=False, transform=None
            )

            aug = T.Compose([
                T.RandomHorizontalFlip(),
                T.RandomRotation(10),
                T.ToTensor(),
                T.Resize((224,224)),
                T.Normalize([0.485,0.456,0.406],
                            [0.229,0.224,0.225])
            ])

            imgs = []
            labels = []

            for idx in chosen:
                img,_ = base_ds[X_client.indices[idx]]
                imgs.append(aug(img))
                labels.append(cls)

            new_X.append(torch.utils.data.TensorDataset(
                torch.stack(imgs),
                torch.tensor(labels)
            ))
            new_y.extend(labels)

    return ConcatDataset(new_X), np.array(new_y)

#**8. Flicker Undersampling – LOCAL Redundancy (UNCHANGED)**

In [ ]:
def get_majority_classes(y, lin_threshold=0.9, n_classes=10):
    cnt = np.bincount(y, minlength=n_classes)
    Lmax = cnt.max()

    majority_classes = [
        c for c in range(n_classes)
        if cnt[c] / Lmax >= lin_threshold
    ]

    return majority_classes

In [ ]:
def local_redundancy_majority(X, y, theta, lin_threshold=0.9):
    maj_classes = get_majority_classes(y, lin_threshold)

    if len(maj_classes) == 0:
        return np.array([]), np.array([])

    maj_idx = np.where(np.isin(y, maj_classes))[0]
    X_maj = Subset(X, maj_idx)

    feats = extract_features_batched(X_maj)
    feats = feats / np.linalg.norm(feats, axis=1, keepdims=True)

    sim = feats @ feats.T
    mean_sim = sim.mean(axis=1)
    var_sim = sim.var(axis=1)

    k = max(1, int(theta * len(mean_sim)))
    buffer_local = np.argsort(mean_sim)[-k:]
    buffer_local = buffer_local[np.argsort(var_sim[buffer_local])[::-1]]

    buffer_idx = maj_idx[buffer_local]

    return buffer_idx, feats[buffer_local]

#**9. Flicker Undersampling – GLOBAL Redundancy (NEW)**

In [ ]:
def flicker_undersample_global(
    dom_id, clients, theta=0.2, eta=0.6, chi=0.3
):
    Xd = clients[dom_id]["X"]
    yd = clients[dom_id]["y"]

    # Step 1: Local redundancy (majority-only)
    buffer_idx, buffer_vecs = local_redundancy_majority(Xd, yd, theta)

    # Step 2: Collect ALL samples from other clients
    other_feats = []

    for cid, client in clients.items():
        if cid == dom_id:
            continue

        feats = extract_features_batched(client["X"])
        feats = feats / np.linalg.norm(feats, axis=1, keepdims=True)
        other_feats.append(feats)

    # Shape: (total_other_samples, feat_dim)
    other_feats = np.vstack(other_feats)
    total_other = len(other_feats)

    # Step 3: Cosine similarity
    sims = cosine_similarity(buffer_vecs, other_feats)
    # Shape: (num_buffer_vectors, total_other_samples)

    # Step 4: Global redundancy check PER BUFFER VECTOR
    remove = []
    threshold = int(np.ceil(chi * total_other))

    for i, idx in enumerate(buffer_idx):
        count_similar = np.sum(sims[i] >= eta)

        if count_similar >= threshold:
            remove.append(idx)

    # Step 5: Remove selected samples
    mask = np.ones(len(Xd), dtype=bool)
    mask[remove] = False

    return Subset(Xd, np.where(mask)[0]), yd[mask]

#**10. Full Training Simulation Loop**

**Defining Max_rounds, Maximum_local_imbalance for each clients**

In [ ]:
MAX_DOM_ROUNDS = 6
LIn_MAX = 0.75   # between 0.70–0.80 

In [ ]:
def compute_LIn(y):
    cnt = np.bincount(y, minlength=10)
    return cnt.min() / cnt.max()

#**Training and Testing Phase**

**Merge the dataset as of now**

In [ ]:
def safe_collate(batch):
    xs, ys = zip(*batch)

    xs = torch.stack(xs)
    ys = torch.tensor(ys, dtype=torch.long)

    return xs, ys

#Load test Data

In [ ]:
test_dataset = dset.CIFAR10(
    root="./data", train=False, download=True,
    transform=transform_resnet
)

test_loader = DataLoader(
    test_dataset, batch_size=256, shuffle=False
)

**Define the Model**

In [ ]:
class CIFARClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone.fc = nn.Identity()

        for p in self.backbone.parameters():
            p.requires_grad = False

        self.head = nn.Linear(512, 10)

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)

**Training Loop**

In [ ]:
def train(model, loader, optimizer, criterion, epochs=10):
    for ep in range(epochs):
        model.train()
        total_loss = 0

        for x, y in loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)

        avg_loss = total_loss / len(loader.dataset)
        print(f"Epoch {ep}: train loss = {avg_loss:.4f}")

**Evaluate the Model**

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

**Optional**

In [ ]:
def per_class_accuracy(model, loader, n_classes=10):
    model.eval()
    correct = np.zeros(n_classes)
    total = np.zeros(n_classes)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)

            for i in range(len(y)):
                total[y[i]] += 1
                correct[y[i]] += (preds[i] == y[i]).item()

    return correct / np.maximum(total, 1)

**Full FLICKER Experiment**

In [ ]:
def run_full_flicker_experiment(run_id, alpha=0.5, seed=None):
    print(f"\n================ RUN {run_id} ================\n")

    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)

    # --------- 1. Create imbalanced clients ----------
    N_CLIENTS = 3
    client_indices = distribute_cifar_imbalanced(
        labels=all_labels,
        n_clients=N_CLIENTS,
        alpha=alpha
    )

    clients = {}
    for cid in range(N_CLIENTS):
        Xc = Subset(full_train_dataset, client_indices[cid])
        yc = all_labels[client_indices[cid]]
        clients[cid] = {"X": Xc, "y": yc}
        print(f"Client {cid} class dist:", Counter(yc))

    # --------- 2. Reset Flicker bookkeeping ----------
    dominance_count = {cid: 0 for cid in clients}
    excluded_clients = set()
    client_resample_flag = {cid: 0 for cid in clients}

    # --------- 3. Flicker rounds ----------
    ROUNDS = 25   # global loop; dominance is client-limited
    for r in range(ROUNDS):
        LDs = {cid: np.bincount(clients[cid]["y"], minlength=10) for cid in clients}
        LDs_norm = {cid: LDs[cid] / np.linalg.norm(LDs[cid]) for cid in LDs}

        GD = sum(LDs.values())
        GD_norm = GD / np.linalg.norm(GD)

        dom, _ = find_dominant_client_with_saturation(
            LDs_norm, GD_norm, clients, excluded_clients
        )

        if dom is None:
            print("No eligible dominant clients left. Stopping.")
            break

        print("Dominant client:", dom)
        Lin_dom = compute_LIn(clients[dom]["y"])

        if dominance_count[dom] >= MAX_DOM_ROUNDS or Lin_dom >= LIn_MAX:
            print(f"Client {dom} saturated (M={dominance_count[dom]}, LIn={Lin_dom:.3f})")
            excluded_clients.add(dom)
            continue

        if client_resample_flag[dom] == 0:
            print("→ Oversampling")
            Xn, yn = flicker_oversample(clients[dom]["X"], clients[dom]["y"])
            client_resample_flag[dom] = 1
        else:
            print("→ Undersampling (Local + Global)")
            Xn, yn = flicker_undersample_global(dom, clients)
            client_resample_flag[dom] = 0

        clients[dom]["X"] = Xn
        clients[dom]["y"] = yn
        dominance_count[dom] += 1
        print("Updated dist:", Counter(yn))

    # --------- 4. Merge final dataset ----------
    final_train_dataset = ConcatDataset(
        [clients[cid]["X"] for cid in clients]
    )

    print("Final training size:", len(final_train_dataset))
    
    final_labels = np.concatenate(
        [clients[cid]["y"] for cid in clients]
    )

    print("Final class distribution:", Counter(final_labels))

    train_loader = DataLoader(
        final_train_dataset,
        batch_size=128,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        collate_fn=safe_collate
    )
    # --------- 5. Train model ----------
    model = CIFARClassifier().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.head.parameters(), lr=1e-3)

    train(model, train_loader, optimizer, criterion, epochs=10)

    # --------- 6. Evaluate ----------
    acc = evaluate(model, test_loader)
    print(f"Run {run_id} Test Accuracy: {acc:.4f}")
    print(f"Per-class accuracy-> {run_id}: {per_class_accuracy(model, test_loader)}")

    return acc


In [ ]:
NUM_RUNS = 6
accuracies = []

for i in range(NUM_RUNS):
    acc = run_full_flicker_experiment(
        run_id=i,
        alpha=0.5,
        seed=100 + i   # different seed → different distribution
    )
    accuracies.append(acc)

print("All accuracies:", accuracies)
print("Mean accuracy:", np.mean(accuracies))
print("Std deviation:", np.std(accuracies))

**Graph(Run vs Accuracy)**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.plot(range(NUM_RUNS), accuracies, marker='o')
plt.xlabel("Run Number")
plt.ylabel("Test Accuracy")
plt.title("Flicker Framework Performance Across Runs")
plt.grid(True)
plt.show()